# Hybrid ModernBERT + SFS: Klasifikasi Sentimen Amazon (3 Kelas)

**Abstrak**
Notebook ini bertujuan untuk menetapkan **Standard Performance** menggunakan arsitektur **ModernBERT** terkini dengan metode *Transfer Learning* yang dimodifikasi menggunakan **Spherical Fuzzy Sets (SFS)** pada dataset *Amazon Product Reviews*.

Eksperimen ini mengevaluasi kemampuan arsitektur **ModernBERT**, sebuah model Transformer canggih, dalam mengklasifikasikan 3 jenis sentimen (Negative, Neutral, Positive) dengan penanganan ketidakpastian menggunakan logika Fuzzy Bola (SFS). Dataset diambil sebanyak 5000 sampel dan dibagi secara manual menjadi **Train (80%)**, **Validation (10%)**, dan **Test (10%)** untuk memastikan evaluasi yang objektif.

**Metodologi**
* **Dataset:** Amazon Product Reviews (60000 Data, 3 Kelas)
* **Split Data:** 80% Training, 10% Validation, 10% Testing (Stratified Split)
* **Arsitektur:** ModernBERT-base + Spherical Fuzzy Decision Module
* **Hyperparameters:** Epoch=5, Batch=32, LR=2e-5 (AdamW)

## 0. Setup Reproducibility (Wajib untuk Jurnal)
Mengunci *Random Seed* agar hasil eksperimen konsisten dan valid saat diuji oleh pihak lain.

In [ ]:
# Install Library Khusus ModernBERT & Tools
!pip install -qU transformers accelerate datasets sentencepiece torchinfo

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    """
    Mengunci semua random state agar eksperimen bisa diulang (Reproducible).
    """
    # 1. Environment Python
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. Random Std Lib & Numpy
    random.seed(seed)
    np.random.seed(seed)
    
    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # 4. CUDNN Backend
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Output Konfirmasi
    print(f"🔒 Random Seed  : {seed}")
    print("✅ Status       : Reproducibility ON (Hasil Konsisten)")

# Panggil fungsi ini PALING AWAL
set_seed(42)

## 1. Import Library & Konfigurasi

In [ ]:
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torchinfo import summary

# --- KONFIGURASI UTAMA ---
DATA_PATH = '/kaggle/input/amazon-product-reviews/Reviews.csv'
MODEL_NAME = 'answerdotai/ModernBERT-base'
CLASSES = ['Negative', 'Neutral', 'Positive'] # 3 Kelas
N_SAMPLES = 60000   # Mengambil 5000 Data
MAX_LEN = 128      # Panjang Token
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- OUTPUT INFORMASI ---
print("✅ Setup Selesai.")
print("-" * 50)
print(f"🚀 Device      : {str(DEVICE).upper()}")
print(f"🧠 Model Base  : {MODEL_NAME}")
print(f"📐 Fuzzy Logic : Spherical Fuzzy Sets (SFS)")
print(f"⚙️  Training    : {EPOCHS} Epochs | Batch Size {BATCH_SIZE} | LR {LR}")
print(f"🏷️  Total Kelas : {len(CLASSES)} -> {CLASSES}")
print("-" * 50)

## 2. Data Engineering: Splitting & Custom Dataset
Dataset ini berupa CSV teks. Kita akan melakukan mapping skor ke 3 kelas, sampling 5000 data, dan splitting manual **80:10:10**.

In [ ]:
# 1. Load & Processing Data
print("📂 Sedang Memproses Dataset Amazon...")
df = pd.read_csv(DATA_PATH)

# Mapping Score ke 3 Label (0: Neg, 1: Neu, 2: Pos)
def map_sentiment(score):
    if score <= 2: return 0 # Negative
    elif score == 3: return 1 # Neutral
    else: return 2 # Positive

df['label'] = df['Score'].apply(map_sentiment)

# Sampling 5000 Data (Stratified agar seimbang)
print(f"   ► Mengambil {N_SAMPLES} Sampel Stratified...")
df_sample = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(int(N_SAMPLES/3)) if len(x) > int(N_SAMPLES/3) else x
)
# Jika data kurang (misal Neutral sedikit), isi sisanya random
if len(df_sample) < N_SAMPLES:
    remaining = N_SAMPLES - len(df_sample)
    df_remain = df.drop(df_sample.index).sample(remaining, random_state=42)
    df_remain['label'] = df_remain['Score'].apply(map_sentiment)
    df_sample = pd.concat([df_sample, df_remain])

texts = df_sample['Text'].values
labels = df_sample['label'].values

# 2. Splitting (80% Train, 10% Val, 10% Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 3. Output Informasi Sederhana & Jelas
print("-" * 80)
print(f"{'KELAS (LABEL)':<35} | {'TRAIN':<8} | {'VAL':<8} | {'TEST':<8} | {'TOTAL':<8}")
print("-" * 80)

for i, class_name in enumerate(CLASSES):
    n_train = (y_train == i).sum()
    n_val = (y_val == i).sum()
    n_test = (y_test == i).sum()
    n_total = n_train + n_val + n_test
    print(f"{class_name:<35} | {n_train:<8} | {n_val:<8} | {n_test:<8} | {n_total:<8}")

print("-" * 80)
print(f"{'TOTAL SELURUHNYA':<35} | {len(X_train):<8} | {len(X_val):<8} | {len(X_test):<8} | {len(df_sample):<8}")
print("-" * 80)
print("✅ Data siap digunakan.")

In [ ]:
# 4. Tokenizer & Custom Dataset
print(f"🔄 Memuat Tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AmazonDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# 5. Buat DataLoader
train_ds = AmazonDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds = AmazonDataset(X_val, y_val, tokenizer, MAX_LEN)
test_ds = AmazonDataset(X_test, y_test, tokenizer, MAX_LEN)

dataloaders = {
    'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    'val': DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
    'test': DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
}

# --- OUTPUT RINGKAS (CHECKER) ---
print("✅ DataLoader Berhasil Dibuat.")
print("-" * 75)
print(f"{'SET':<10} | {'TOTAL DATA':<15} | {'BATCH SIZE':<12} | {'BATCHES':<15} | {'SHUFFLE'}")
print("-" * 75)
for phase in ['train', 'val', 'test']:
    print(f"{phase.upper():<10} | {len(dataloaders[phase].dataset):<15} | {BATCH_SIZE:<12} | {len(dataloaders[phase]):<15} | {'✅ Yes' if phase=='train' else '❌ No'}")

# Cek batch sample
sample = next(iter(dataloaders['train']))
print("-" * 75)
print(f"🔍 Cek Integritas Tensor (Sample Batch):")
print(f"   ► Input IDs Shape : {sample['input_ids'].shape}")
print(f"   ► Label Shape     : {sample['label'].shape}")
print("-" * 75)

## 3. Definisi Model: ModernBERT + SFS Module

**Analisis Arsitektur:**
Kami menggunakan **ModernBERT** sebagai backbone. Layer klasifikasi standar digantikan dengan custom head yang output-nya (3 logit) akan diproses menggunakan logika **Spherical Fuzzy Sets (SFS)**.

**Spherical Fuzzy Sets (SFS):**
* **Non-Membership ($\nu$):** Probabilitas Negative.
* **Hesitancy ($\pi$):** Probabilitas Neutral.
* **Membership ($\mu$):** Probabilitas Positive.
* **Score:** $\mu^2 - \nu^2$ (Defuzzifikasi).

In [ ]:
# --- 1. Spherical Fuzzy Sets Logic ---
class SphericalFuzzyLogic:
    def __init__(self):
        pass
        
    def decide(self, probs):
        """
        Input: List/Array [prob_neg, prob_neu, prob_pos]
        Output: Predicted Class Index (0, 1, or 2)
        """
        # Mapping Softmax ke SFS Parameters
        nu = probs[0]  # Negative (Non-Membership)
        pi = probs[1]  # Neutral (Hesitancy)
        mu = probs[2]  # Positive (Membership)
        
        # Hitung SFS Score (Improved Score Function)
        score = (mu ** 2) - (nu ** 2)
        
        # Logika Keputusan SFS
        # Rule: Jika Hesitancy (Netral) sangat dominan (> Pos & > Neg), maka Netral
        if pi > mu and pi > nu:
            return 1 # Neutral
        
        # Rule: Berdasarkan Score
        if score > 0:
            return 2 # Positive
        else:
            return 0 # Negative

# --- 2. Hybrid Model Definition ---
class ModernBERTSpherical(nn.Module):
    def __init__(self, model_name, num_classes=3):
        super(ModernBERTSpherical, self).__init__()
        # Load Body Only
        self.bert = AutoModel.from_pretrained(model_name)
        self.config = AutoConfig.from_pretrained(model_name)
        
        # Custom Head (3 Outputs)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(self.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Ambil representasi kalimat (CLS token)
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        output = self.drop(pooled_output)
        logits = self.out(output)
        return logits

# Inisialisasi Model
model = ModernBERTSpherical(MODEL_NAME, num_classes=3)
model = model.to(DEVICE)

# Hitung Parameter
total_params = sum(p.numel() for p in model.parameters())
print("-" * 55)
print(f"✅ MODEL HYBRID TERBANGUN")
print(f"🧠 Backbone     : ModernBERT (Transfer Learning)")
print(f"🎯 Head Type    : Linear -> SFS Decision (3 Classes)")
print(f"🎛️ Total Params : {total_params:,}")
print("-" * 55)

## 4. Analisis Kompleksitas Model (Penting untuk Jurnal)

In [ ]:
print("📊 Analisis Arsitektur ModernBERT Hybrid")
# Dummy Input untuk Torchinfo
dummy_ids = torch.randint(0, 1000, (1, MAX_LEN), dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones((1, MAX_LEN), dtype=torch.long).to(DEVICE)

summary(model, input_data=[dummy_ids, dummy_mask], 
        col_names=["input_size", "output_size", "num_params", "mult_adds"], 
        depth=2)

## 5. Training Loop
Training dilakukan menggunakan `CrossEntropyLoss` standar untuk mengoptimalkan probabilitas. Logika SFS diterapkan saat evaluasi/inferensi.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

def train_model(model, num_epochs):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for d in tqdm(dataloaders[phase], desc=phase, leave=False):
                input_ids = d['input_ids'].to(DEVICE)
                attention_mask = d['attention_mask'].to(DEVICE)
                targets = d['label'].to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(input_ids, attention_mask)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, targets)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * input_ids.size(0)
                running_corrects += torch.sum(preds == targets.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                scheduler.step(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = model.state_dict()
                    torch.save(model.state_dict(), 'modernbert_sfs_best.pth')

    print('\n' + '='*30)
    print('       FINAL BEST RESULTS       ')
    print('='*30)
    print(f"🔥 Best Training Acc   : {max(history['train_acc']):.4f}")
    print(f"💎 Best Validation Acc : {best_acc:.4f}")
    print('='*30 + '\n')

    model.load_state_dict(best_model_wts)
    return model, history

# Jalankan Training
model, history = train_model(model, EPOCHS)

## 6. Visualisasi & Evaluasi Akhir (Jumbo HD)

In [ ]:
# Visualisasi Training Curve (Style Jumbo HD)
epochs_range = range(1, len(history['train_acc']) + 1)

plt.figure(figsize=(18, 12))
plt.grid(True, which='both', linestyle='--', linewidth=3.0, alpha=0.8, color='gray')

lw_size = 4.5
mk_size = 15

plt.plot(epochs_range, history['train_acc'], 'o-', label='Training Accuracy', 
         color='#1f77b4', markersize=mk_size, linewidth=lw_size)
plt.plot(epochs_range, history['val_acc'], 's-', label='Validation Accuracy', 
         color='#ff7f0e', markersize=mk_size, linewidth=lw_size)
plt.plot(epochs_range, history['train_loss'], '^-', label='Training Loss', 
         color='#2ca02c', markersize=mk_size, linewidth=lw_size)
plt.plot(epochs_range, history['val_loss'], 'd-', label='Validation Loss', 
         color='#d62728', markersize=mk_size, linewidth=lw_size)

plt.title('Training Result: ModernBERT + SFS (3 Classes)', fontsize=40, fontweight='bold', pad=25)
plt.xlabel('Epoch', fontsize=40, labelpad=15)
plt.ylabel('Accuracy / Loss', fontsize=40, labelpad=15)
plt.xticks(fontsize=35)
plt.yticks(fontsize=35)
plt.legend(loc='center right', fontsize=30, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig('modernbert_sfs_training_hd.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ==============================================================================
# VISUALISASI TERPISAH (AKURASI & LOSS) - STYLE JUMBO HD
# ==============================================================================
def plot_curves_separated(history):
    # Setup Data
    train_acc = history['train_acc']
    val_acc = history['val_acc']
    train_loss = history['train_loss']
    val_loss = history['val_loss']
    epochs = range(1, len(train_acc) + 1)
    
    # 1. PLOT AKURASI (ACCURACY CURVE)
    plt.figure(figsize=(12, 8))
    # Grid di belakang
    plt.grid(True, which='major', linestyle='--', linewidth=1.5, alpha=0.6, color='gray')
    
    plt.plot(epochs, train_acc, label='Training Accuracy', color='#1f77b4', linewidth=4, marker='o', markersize=10)
    plt.plot(epochs, val_acc, label='Validation Accuracy', color='#ff7f0e', linewidth=4, marker='s', markersize=10)
    
    plt.title('Model Accuracy: ModernBERT + SFS', fontsize=24, fontweight='bold', pad=20)
    plt.xlabel('Epoch', fontsize=20, fontweight='bold', labelpad=15)
    plt.ylabel('Accuracy', fontsize=20, fontweight='bold', labelpad=15)
    
    plt.xticks(epochs, fontsize=16, fontweight='bold')
    plt.yticks(fontsize=16, fontweight='bold')
    plt.legend(fontsize=16, loc='lower right', frameon=True, shadow=True)
    
    # Format Y-Axis agar 4 angka belakang koma
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
    
    plt.tight_layout()
    plt.savefig('modernbert_accuracy_curve_hd.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Grafik Akurasi disimpan: modernbert_accuracy_curve_hd.png")
    print("-" * 60)

    # 2. PLOT LOSS (LOSS CURVE)
    plt.figure(figsize=(12, 8))
    plt.grid(True, which='major', linestyle='--', linewidth=1.5, alpha=0.6, color='gray')
    
    plt.plot(epochs, train_loss, label='Training Loss', color='#2ca02c', linewidth=4, marker='^', markersize=10)
    plt.plot(epochs, val_loss, label='Validation Loss', color='#d62728', linewidth=4, marker='d', markersize=10)
    
    plt.title('Model Loss: ModernBERT + SFS', fontsize=24, fontweight='bold', pad=20)
    plt.xlabel('Epoch', fontsize=20, fontweight='bold', labelpad=15)
    plt.ylabel('Loss', fontsize=20, fontweight='bold', labelpad=15)
    
    plt.xticks(epochs, fontsize=16, fontweight='bold')
    plt.yticks(fontsize=16, fontweight='bold')
    plt.legend(fontsize=16, loc='upper right', frameon=True, shadow=True)
    
    ax = plt.gca()
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
    
    plt.tight_layout()
    plt.savefig('modernbert_loss_curve_hd.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Grafik Loss disimpan: modernbert_loss_curve_hd.png")

# Jalankan Fungsi
plot_curves_separated(history)

In [ ]:
# Evaluasi dengan Spherical Fuzzy Logic
sfs_logic = SphericalFuzzyLogic()
model.eval()
all_preds_sfs = []
all_labels = []

print("🚀 Memulai Evaluasi SFS pada Test Set...")
start_time = time.time()

with torch.no_grad():
    for d in tqdm(dataloaders['test'], desc="Testing"):
        input_ids = d['input_ids'].to(DEVICE)
        attention_mask = d['attention_mask'].to(DEVICE)
        targets = d['label'].to(DEVICE)
        
        logits = model(input_ids, attention_mask)
        probs = torch.nn.functional.softmax(logits, dim=1)
        
        # Loop batch untuk SFS Logic
        for i in range(len(probs)):
            p = probs[i].cpu().numpy()
            # Apply SFS Decision Rule
            decision = sfs_logic.decide(p)
            all_preds_sfs.append(decision)
            
        all_labels.extend(targets.cpu().numpy())

end_time = time.time()
inference_time = end_time - start_time
avg_time_ms = (inference_time / len(dataloaders['test'].dataset)) * 1000
print(f"⏱️ Rata-rata inferensi per teks : {avg_time_ms:.4f} ms")

# Classification Report
print("\n📄 Classification Report (ModernBERT + SFS):")
print(classification_report(all_labels, all_preds_sfs, target_names=CLASSES, digits=4))

# Visualisasi Confusion Matrix Jumbo HD
cm = confusion_matrix(all_labels, all_preds_sfs)

TITLE_SIZE = 20      
LABEL_SIZE = 18      
TICK_SIZE = 16        
ANNOT_SIZE = 48       
CBAR_SIZE = 14        

plt.figure(figsize=(10, 8))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                 xticklabels=CLASSES, yticklabels=CLASSES,
                 annot_kws={"size": ANNOT_SIZE, "weight": "bold"}) 

plt.title('Confusion Matrix: ModernBERT + SFS', fontsize=TITLE_SIZE, fontweight='bold', pad=25)
plt.xlabel('Predicted Label', fontsize=LABEL_SIZE, fontweight='bold', labelpad=15)
plt.ylabel('True Label', fontsize=LABEL_SIZE, fontweight='bold', labelpad=15)
plt.xticks(fontsize=TICK_SIZE, fontweight='bold', rotation=45, ha='right')
plt.yticks(fontsize=TICK_SIZE, fontweight='bold', rotation=0)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=CBAR_SIZE)
plt.tight_layout()
plt.savefig('modernbert_sfs_cm_hd.png', dpi=300, bbox_inches='tight')
print(f"\n✅ Gambar Confusion Matrix disimpan.")
plt.show()